In [ ]:
# 1) backend/ → 供 llm.py 内 `from config import ...`
# 2) backend/core/ → 供笔记本里 `from llm import ...`（与当前工作目录无关）
import sys
from pathlib import Path

_p = Path.cwd().resolve()
backend = None
for _ in range(10):
    if (_p / "config.py").exists():
        backend = _p
        break
    _p = _p.parent
if backend is None:
    raise RuntimeError("找不到 backend（含 config.py 的目录）。请从项目内打开笔记本，或 cd 到 backend 再启动 Jupyter。")
sys.path.insert(0, str(backend))
sys.path.insert(0, str(backend / "core"))


In [8]:

import asyncio
import time
from dataclasses import dataclass
from statistics import mean


# 这个例程不直接打真实 LLM，而是“模拟耗时结构”：
# - old_*：在 async 函数里直接做阻塞操作（等价以前的同步 invoke / 同步 retrieve）
# - new_*：把阻塞操作放到 asyncio.to_thread，并加 semaphore 限流

def _fake_blocking_retrieve(delay_sec: float = 0.08) -> list[str]:
    time.sleep(delay_sec)
    return ["chunk-a", "chunk-b", "chunk-c"]


def _fake_blocking_llm_invoke(delay_sec: float = 0.35) -> str:
    time.sleep(delay_sec)
    return "ok"


@dataclass
class BenchConfig:
    concurrency: int
    rounds: int = 1
    llm_cap: int = 10

In [10]:
async def old_pipeline(req_id: int) -> dict:
    """老方法：async 函数内直接阻塞，事件循环被卡住。"""
    _ = _fake_blocking_retrieve(0.08)
    _ = _fake_blocking_llm_invoke(0.35)
    return {"req_id": req_id, "ok": True}


async def new_pipeline(req_id: int, sem: asyncio.Semaphore) -> dict:
    """新方法：阻塞段放到线程池，LLM 调用受 semaphore 保护。"""
    _ = await asyncio.to_thread(_fake_blocking_retrieve, 0.08)
    async with sem:
        _ = await asyncio.to_thread(_fake_blocking_llm_invoke, 0.35)
    return {"req_id": req_id, "ok": True}


async def _run_once_old(concurrency: int) -> float:
    t0 = time.perf_counter()
    await asyncio.gather(*[old_pipeline(i) for i in range(concurrency)])
    return time.perf_counter() - t0


async def _run_once_new(concurrency: int, llm_cap: int) -> float:
    sem = asyncio.Semaphore(llm_cap)
    t0 = time.perf_counter()
    await asyncio.gather(*[new_pipeline(i, sem) for i in range(concurrency)])
    return time.perf_counter() - t0


async def benchmark(config: BenchConfig) -> dict:
    old_costs = [await _run_once_old(config.concurrency) for _ in range(config.rounds)]
    new_costs = [await _run_once_new(config.concurrency, config.llm_cap) for _ in range(config.rounds)]

    old_avg = mean(old_costs)
    new_avg = mean(new_costs)
    return {
        "concurrency": config.concurrency,
        "rounds": config.rounds,
        "old_avg_sec": old_avg,
        "new_avg_sec": new_avg,
        "speedup": old_avg / new_avg if new_avg > 0 else float("inf"),
        "old_qps": config.concurrency / old_avg if old_avg > 0 else 0.0,
        "new_qps": config.concurrency / new_avg if new_avg > 0 else 0.0,
    }

In [6]:
import json
raw='{{"intent":"knowledge"}}'
obj = json.loads(raw)
print(raw)
print(obj)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [ ]:
lock = asyncio.Lock()

# 协程A
async with lock:
    print("A 进入")
    await asyncio.sleep(3)
    print("A 离开")

# 协程B（几乎同时到）
async with lock:
    print("B 进入")  # 会在 A 离开后才打印

A 进入
A 离开
B 进入


: 